In [ ]:
from adaptive_latents import datasets
import matplotlib.pyplot as plt
import numpy as np
import pathlib
import cv2
import tifffile
from scipy.ndimage import gaussian_filter


In [ ]:
d = datasets.Zong22Dataset(sub_dataset_identifier=10)
d1 = datasets.Zong22Dataset(sub_dataset_identifier=10)
d2 = datasets.Zong22Dataset(sub_dataset_identifier=11)
d.sub_datset_info.loc[d.sub_dataset,'basepath']

In [ ]:

row = d.sub_datset_info.loc[d.sub_dataset]
if not np.isnan(row.behavior_video):
    fpath = (d.dataset_base_path / row.basepath / row.behavior_video).resolve()
    cap = cv2.VideoCapture(fpath)
    int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

In [ ]:
tiff = tifffile.imread(d.dataset_base_path/ row.basepath / row.raw_frames)
N = tiff.shape[0]
tiff.shape

In [ ]:
F = np.load(d.dataset_base_path/ row.basepath / 'suite2p'/'plane0' / 'F.npy')

F.shape

In [ ]:
%matplotlib inline
plt.plot(d.F_all[0:10,:].T)
# plt.axvline(x=F.shape[1]//2, color='k')

In [ ]:
plt.imshow(d.F_all[:,N-200:N+200].T, aspect='auto')

In [ ]:
combined = np.vstack([d1.behavioral_data, d2.behavioral_data])
plt.plot(combined[N-100:N+100,:])

In [ ]:
row

In [ ]:
d = datasets.Zong22Dataset(sub_dataset_identifier=3)
d.sub_datset_info


In [ ]:
for i in range(7,13):
    d = datasets.Zong22Dataset(sub_dataset_identifier=i)

    row = d.sub_datset_info.loc[d.sub_dataset]
    recording_id = row.basepath.replace('/', '_') + f"session_{row.part_of_F[0]}_of_{row.part_of_F[1]}"

    data = {
        'behavior_t': np.array(d.behavior_df['t']),
        'behavior_position_x': np.array(d.behavior_df['x']),
        'behavior_position_y': np.array(d.behavior_df['y']),
        'behavior_head_direction': np.array(d.behavior_df['hd']),
        'florescence_t': np.array(d.neural_data.t),
        'florescence_df_over_f': np.array(d.neural_data),
        'florescence_raw': np.array(d.F),
        'recording_id': recording_id,
    }


    np.savez(f'/home/jgould/Downloads/{recording_id}.npz', **data)
del data

In [ ]:
data = np.load('/home/jgould/Downloads/MEC_recordings_97045_20210305_session_5_of_6.npz')


In [ ]:
data['florescence_df_over_f'].shape

In [ ]:
len(d.neural_data.t), len(d.behavioral_data.t)

In [ ]:
d.neural_data.t, d.behavioral_data.t

In [ ]:
for i in range(6):
    data = np.load(f'/home/jgould/Downloads/MEC_recordings_97045_20210305_session_{i+1}_of_6.npz')
    plt.scatter(data['behavior_position_x'][::2], data['behavior_position_y'][::2],c=data['florescence_df_over_f'][:,1])

In [ ]:

# please visualize the heatmap of neuron 1's activiy across space
for i in range(6):
    data = np.load(f'/home/jgould/Downloads/MEC_recordings_97045_20210305_session_{i+1}_of_6.npz')
    plt.scatter(data['behavior_position_x'][::2], data['behavior_position_y'][::2],c=data['florescence_df_over_f'][:,1])


In [ ]:
# Heatmap of neuron 1's activity across space (aggregated across all sessions)
n_bins = 50
neuron_idx = 1

# Collect all positions and activity across sessions
all_x, all_y, all_activity = [], [], []
for i in range(6):
    data = np.load(f'/home/jgould/Downloads/MEC_recordings_97045_20210305_session_{i+1}_of_6.npz')
    x = data['behavior_position_x'][::2]
    y = data['behavior_position_y'][::2]
    activity = data['florescence_df_over_f'][:, neuron_idx]
    min_len = min(len(x), len(activity))
    all_x.append(x[:min_len])
    all_y.append(y[:min_len])
    all_activity.append(activity[:min_len])


all_x = np.concatenate(all_x)
all_y = np.concatenate(all_y)
all_activity = np.concatenate(all_activity)


In [ ]:


# 2D bin the activity
x_edges = np.linspace(np.nanmin(all_x), np.nanmax(all_x), n_bins + 1)
y_edges = np.linspace(np.nanmin(all_y), np.nanmax(all_y), n_bins + 1)

activity_sum, _, _ = np.histogram2d(all_x, all_y, bins=[x_edges, y_edges], weights=all_activity)
counts, _, _ = np.histogram2d(all_x, all_y, bins=[x_edges, y_edges])

In [ ]:

with np.errstate(invalid='ignore'):
    smoothed_sum = gaussian_filter(activity_sum, sigma=2)
    smoothed_counts = gaussian_filter(counts, sigma=2)
    heatmap = smoothed_sum / smoothed_counts  # mean activity per bin, smoothed

heatmap = np.nan_to_num(heatmap, nan=0.0)

fig, ax = plt.subplots(figsize=(7, 6))
im = ax.pcolormesh(x_edges, y_edges, heatmap.T, cmap='hot', shading='auto')
fig.colorbar(im, ax=ax, label='Mean ΔF/F')
ax.set_xlabel('Position X')
ax.set_ylabel('Position Y')
ax.set_title(f'Neuron {neuron_idx} Activity Heatmap')
plt.tight_layout()
plt.show()
